# 02. Dataset Schema and EDA

- Parent issue: `#17`
- Status: `active`
- Summary: Explain the dataset shape, category mix, and normalization contract before code paths hard-code assumptions.


## Audience and Why It Matters

Data contributors, reviewers, and non-technical readers who need to see what the benchmark actually looks like.


## Decision / Hypothesis

Normalize all dataset sources into the shared `ReasoningExample` contract and emphasize category-specific task shape over generic math-only framing.


## Environment and Reproduction

- Python: 3.7.9 (tested); 3.11+ recommended for full src/ compatibility
- Run from repo root: `jupyter notebook` from `CS4650-Nvidia-Nemotron-Challenge/`
- Notebook uses `Path.cwd().parent` to resolve repo root when launched from `notebooks/`
- Input: `data/raw/train.csv` (download from Kaggle competition data tab after joining)
- Output: `data/eval/train_normalized.jsonl` (git-ignored, input for notebook 03)
- Companion issue and registry entry: `docs/execution/NOTEBOOKS.md`

In [1]:
import sys
import platform
from pathlib import Path

_cwd = Path.cwd()
REPO_ROOT = _cwd
for _p in [_cwd] + list(_cwd.parents):
    if (_p / "src").is_dir() and (_p / "notebooks").is_dir():
        REPO_ROOT = _p
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root : {REPO_ROOT}")
print(f"Python    : {sys.version.split()[0]}")
print(f"Platform  : {platform.platform()}")

Repo root : C:\Users\SBaroni\CS4650-Nvidia-Nemotron-Challenge
Python    : 3.7.9
Platform  : Windows-10-10.0.26100-SP0


## Method and Outputs

### Planned method
- Inspect the official dataset or mirror columns and sample prompts.
- Document category distribution and the input/output format.
- Show how raw records map into the canonical schema.

### Expected outputs
 - Schema mapping examples
- Category inventory
- Normalization checklist


In [2]:
import pandas as pd
from pathlib import Path

REPO_ROOT = Path.cwd().parent
df = pd.read_csv(REPO_ROOT / "data" / "raw" / "train.csv")

print(f"Shape   : {df.shape}")
print(f"Columns : {df.columns.tolist()}")
print(f"\nFirst row:")
for col, val in df.iloc[0].items():
    print(f"  {col:<20}: {repr(str(val)[:120])}")

Shape   : (9500, 3)
Columns : ['id', 'prompt', 'answer']

First row:
  id                  : '00066667'
  prompt              : "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves opera"
  answer              : '10010111'


In [3]:
# Look at 10 sample prompts to understand the structure
for i in range(10):
    print(f"--- Row {i} ---")
    print(df.iloc[i]["prompt"][:200])
    print(f"answer: {df.iloc[i]['answer']}")
    print()

--- Row 0 ---
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or ch
answer: 10010111

--- Row 1 ---
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or ch
answer: 01000011

--- Row 2 ---
In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:
ucoov pwgtfyoqg vorq yrjjoe -> queen discovers near valley
pqrsfv pqorzg wvgwpo trgbjo -> dragon dreams inside 
answer: cat imagines book

--- Row 3 ---
In Alice's Wonderland, numbers are secretly converted into a different numeral system. Some examples are given below:
11 -> XI
15 -> XV
94 -> XCIV
19 -> XIX
Now, write the number 38 in the Wonderland 
answer: XXXVIII

--- Row 4 ---
In Alice's Wonderland, secret encryption rules

In [4]:
import re

def extract_category(prompt):
    p = str(prompt)
    if "bit manipulation" in p:
        return "bit_manipulation"
    elif "encryption rules" in p:
        return "text_cipher"
    elif "numeral system" in p:
        return "numeral_system"
    elif "unit conversion" in p:
        return "unit_conversion"
    elif "gravitational constant" in p:
        return "physics_gravity"
    elif "transformation rules is applied to equations" in p:
        return "equation_symbolic"
    elif "algebraic" in p.lower() or "solve for" in p.lower():
        return "equation_numeric"
    else:
        return "unknown"

df["category"] = df["prompt"].apply(extract_category)

print("=== Category Distribution ===")
print(df["category"].value_counts())
print(f"\nUnknown rows: {(df['category'] == 'unknown').sum()}")

=== Category Distribution ===
bit_manipulation     1602
physics_gravity      1597
unit_conversion      1594
text_cipher          1576
numeral_system       1576
equation_symbolic    1555
Name: category, dtype: int64

Unknown rows: 0


In [5]:
df["prompt_len"] = df["prompt"].str.len()
df["answer_len"] = df["answer"].str.len()

print("=== Prompt Length ===")
print(df["prompt_len"].describe().round(1))

print("\n=== Answer Length ===")
print(df["answer_len"].describe().round(1))

print("\n=== Missing Values ===")
print(df.isnull().sum())

=== Prompt Length ===
count    9500.0
mean      301.5
std       104.3
min       177.0
25%       209.0
50%       281.0
75%       371.0
max       510.0
Name: prompt_len, dtype: float64

=== Answer Length ===
count    9500.0
mean        8.4
std         8.0
min         1.0
25%         4.0
50%         5.0
75%         8.0
max        39.0
Name: answer_len, dtype: float64

=== Missing Values ===
id            0
prompt        0
answer        0
category      0
prompt_len    0
answer_len    0
dtype: int64


In [6]:
has_boxed = df["answer"].str.contains(r"\\boxed", na=False).sum()
print(f"Answers containing \\boxed{{}}: {has_boxed} / {len(df)}")

print("\nSample answers by category:")
for cat, group in df.groupby("category"):
    print(f"  {cat:<25}: {repr(group.iloc[0]['answer'])}")

Answers containing \boxed{}: 0 / 9500

Sample answers by category:
  bit_manipulation         : '10010111'
  equation_symbolic        : '@&'
  numeral_system           : 'XXXVIII'
  physics_gravity          : '154.62'
  text_cipher              : 'cat imagines book'
  unit_conversion          : '16.65'


In [7]:
import json

DATASET_VERSION = "kaggle-v1"
OUT_DIR = REPO_ROOT / "data" / "eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

records = []
for _, row in df.iterrows():
    records.append({
        "example_id": str(row["id"]),
        "category": row["category"],
        "prompt": row["prompt"],
        "gold": str(row["answer"]),
        "source": "kaggle-official",
        "split": "train",
        "dataset_version": DATASET_VERSION,
        "metadata": {}
    })

out_path = OUT_DIR / "train_normalized.jsonl"
with open(out_path, "w") as f:
    for rec in records:
        f.write(json.dumps(rec) + "\n")

print(f"Wrote {len(records)} rows → data/eval/train_normalized.jsonl")
print(f"Sample: {records[0]}")

Wrote 9500 rows → data/eval/train_normalized.jsonl
Sample: {'example_id': '00066667', 'category': 'bit_manipulation', 'prompt': "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n01010001 -> 11011101\n00001001 -> 01101101\n00010101 -> 01010101\n11111111 -> 10000001\n10011101 -> 01000101\n00111011 -> 00001001\n10111101 -> 00000101\n00100110 -> 10110011\n\nNow, determine the output for: 00110100", 'gold': '10010111', 'source': 'kaggle-official', 'split': 'train', 'dataset_version': 'kaggle-v1', 'metadata': {}}


## Results / Open Risks

- Dataset loaded from Kaggle official source (`kaggle-v1`): 9,500 rows, 3 columns (`id`, `prompt`, `answer`).
- 6 categories parsed from prompt text (no `category` column in raw data): `bit_manipulation`, `text_cipher`, `numeral_system`, `unit_conversion`, `physics_gravity`, `equation_symbolic`.
- Category distribution is even (~1,550–1,600 rows each). No missing values.
- Gold answers are plain strings — no `\boxed{}` in training data. The evaluator wraps model output in `\boxed{}` and strips it before exact-match comparison.
- Prompt lengths: mean 301 chars, max 510. Answer lengths: mean 8 chars, max 39.
- Wrote `data/eval/train_normalized.jsonl` (9,500 rows, git-ignored). Input for notebook 03.
- Risk: category parsing uses keyword matching on prompt text — if competition adds new category types, `extract_category()` will return `unknown` and need updating.

## Sources

- [Competition facts doc](../docs/architecture/COMPETITION.md)
- [Architecture doc](../docs/architecture/ARCHITECTURE.md)
- [Kaggle competition data](https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/data)
- [src/contracts.py](../src/contracts.py)